In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Step 41 — RavKav 2025: journeys chained from the taps, on their own inferred alightings (task C9)

The 2025 extracts (step 34, `RavKav_2025_boardings_matrix.ipynb`) carry boardings only: no
alightings, and the `JourneyTransfer` tag flags only 3.7% of taps as a transfer, far below the
2022 files' own `bus_trip_id` linkage (1.52 legs per journey, a third of boardings transfer
legs). Step 34's bus/Metronit destinations are therefore borrowed from the 2022 alighting
pattern. This step gives the 2025 layer an alighting inference and a journey chaining of its
own, reproducing the *definition* step 8 used in 2022 (a journey groups every leg a card takes
without a long enough gap to be a separate trip) on data that lacks the operator-supplied
`bus_trip_id` field the 2022 extract had (`METHODOLOGY.md` §6af "Re-anchoring", reason (ii)).

**Method.** Per (card, date), sort taps by time. A tap's alighting is the location of the
card's *next* tap that day (the "same line direction" refinement of the plan is not built —
there is no line-shape matching here — so this is the plan's method with that one
simplification stated up front), accepted only if the next tap is within 90 minutes and 20 km
(the plan's own plausibility bounds; no separate fare-transfer-window figure exists elsewhere
in this repository, so the same 90-minute bound serves both the alighting inference and the
journey-chaining decision below). The last tap of the day takes the day's first tap's own stop
under the same 20 km bound (the return-home rule); "on a plausible line" beyond that distance
check is not evaluated (no line data at this level). Two consecutive taps chain into one
journey when the same two bounds hold between them — a transfer under this rule is, by
construction, always to the immediately preceding leg's own alighting point, since that point
*is* the next tap's location; the bounds are what actually decide whether a chain continues,
not distance from the alighting (which is trivially satisfied for consecutive taps).

In [2]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9'
OUT = 'Output/ravkav_2025'; os.makedirs(OUT, exist_ok=True); os.makedirs('Output/figures', exist_ok=True)
RK = 'Input/BusRavKav/2025'
USECOLS = ['CardIDbi', 'ClusterName', 'StopCode', 'TransactionDate', 'TransactionTime', 'JourneyTransfer', 'PassengersNumber', 'Weekday']
TRANSFER_WINDOW_MIN = 90   # task C9: reused for both the alighting inference and the chain decision (stated simplification, see markdown above)
MAX_ALIGHT_KM = 20

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
for f in [f'{RK}/Buses_RavKav.csv', f'{RK}/Metronit_RavKav_Data.csv']:
    assert not is_pointer(f), f'{f} is a Git LFS pointer -- python3 tools/lfs_pull.py {f} first'

## 1. Stop locations and representative dates, reused from step 34

In [3]:
# reuse step 34's own (cluster, code) -> (lat, lon, TAZ) resolution and its 42 representative
# Tuesdays, rather than rescanning the raw files for the same answer
located = pd.read_csv(f'{OUT}/stops_located_by_cluster_2025.csv')
north_file = pd.read_csv(f'{OUT}/stops_north_file_taz.csv').set_index('StopCode')
north_share = located.groupby('ClusterName')['TAZ'].apply(lambda t: t.notna().mean())
northern_clusters = set(north_share[north_share >= 0.5].index)
KEY_XY = {(r.ClusterName, r.StopCode): (r.Lat, r.Long, r.TAZ) for r in located.itertuples() if pd.notna(r.TAZ)}
KEY_OUT = {(r.ClusterName, r.StopCode) for r in located.itertuples() if pd.isna(r.TAZ)}

def key_to_xy(cluster, code):
    """(lat, lon, TAZ) of a (cluster, code) inside the study area, else None -- same precedence as step 34's key_to_taz."""
    k = (cluster, code)
    if k in KEY_XY: return KEY_XY[k]
    if k in KEY_OUT: return None
    if cluster in northern_clusters and code in north_file.index and pd.notna(north_file.at[code, 'TAZ']):
        r = north_file.loc[code]; return (r['Lat'], r['Long'], r['TAZ'])
    return None

daily = pd.read_csv(f'{OUT}/daily_totals_by_date.csv')
DATES = sorted(daily.loc[daily['kept (all three)'], 'TransactionDate'])
print(f"{len(DATES)} representative Tuesdays reused from step 34 ({DATES[0]} .. {DATES[-1]})")

42 representative Tuesdays reused from step 34 (2025-01-07 .. 2025-12-30)


## 2. Every located bus / Metronit tap on the 42 representative Tuesdays

In [4]:
def read_taps(path):
    parts = []
    for ch in pd.read_csv(path, usecols=USECOLS, chunksize=2_000_000, dtype={'TransactionTime': str}):
        ch = ch[(ch['Weekday'] == 3) & ch['TransactionDate'].isin(DATES) & (ch['TransactionTime'].str.slice(0, 2).isin(['06', '07', '08'])) & (ch['PassengersNumber'] > 0)]
        if not len(ch): continue
        xy = [key_to_xy(c, s) for c, s in zip(ch['ClusterName'], ch['StopCode'])]
        keep = [v is not None for v in xy]
        ch = ch[keep].copy(); xy = [v for v, k in zip(xy, keep) if k]
        ch['lat'] = [v[0] for v in xy]; ch['lon'] = [v[1] for v in xy]; ch['TAZ'] = [v[2] for v in xy]
        parts.append(ch)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=USECOLS + ['lat', 'lon', 'TAZ'])

t0 = time.time()
bus = read_taps(f'{RK}/Buses_RavKav.csv')
met = read_taps(f'{RK}/Metronit_RavKav_Data.csv')
taps = pd.concat([bus, met], ignore_index=True)
taps['t'] = pd.to_timedelta(taps['TransactionTime'] + ':00')
taps = taps.sort_values(['CardIDbi', 'TransactionDate', 't']).reset_index(drop=True)
print(f"{len(taps):,} bus + Metronit taps located in the study area on the {len(DATES)} representative Tuesdays "
      f"(bus {len(bus):,}, Metronit {len(met):,})  [{time.time() - t0:.0f} s]")

6,015,350 bus + Metronit taps located in the study area on the 42 representative Tuesdays (bus 5,424,117, Metronit 591,233)  [58 s]


## 3. Alighting inference and journey chaining, vectorised by (card, date)

In [5]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

g = taps.groupby(['CardIDbi', 'TransactionDate'], sort=False)
taps['next_lat'] = g['lat'].shift(-1); taps['next_lon'] = g['lon'].shift(-1); taps['next_t'] = g['t'].shift(-1)
taps['next_TAZ'] = g['TAZ'].shift(-1); taps['next_StopCode'] = g['StopCode'].shift(-1)
taps['first_lat'] = g['lat'].transform('first'); taps['first_lon'] = g['lon'].transform('first'); taps['first_TAZ'] = g['TAZ'].transform('first')
taps['group_size'] = g['lat'].transform('size')
taps['is_last'] = taps['next_lat'].isna()
taps['gap_min'] = (taps['next_t'] - taps['t']).dt.total_seconds() / 60
taps['dist_next_km'] = haversine_km(taps['lat'], taps['lon'], taps['next_lat'], taps['next_lon'])
taps['dist_first_km'] = haversine_km(taps['lat'], taps['lon'], taps['first_lat'], taps['first_lon'])

plausible_next = (taps['gap_min'] <= TRANSFER_WINDOW_MIN) & (taps['dist_next_km'] <= MAX_ALIGHT_KM)
# the return-home rule needs a *different* earlier tap that day to compare against -- a card with
# only one tap in the window has nothing to infer from and is correctly left unallocated, not
# "alighted" at its own boarding stop (a bug caught in testing: dist_first_km is trivially 0 for
# a single-tap group, since first == the tap itself)
plausible_return = (taps['group_size'] > 1) & (taps['dist_first_km'] <= MAX_ALIGHT_KM)

taps['alight_TAZ'] = np.where(~taps['is_last'] & plausible_next, taps['next_TAZ'],
                      np.where(taps['is_last'] & plausible_return, taps['first_TAZ'], np.nan))
taps['alight_rule'] = np.where(~taps['is_last'] & plausible_next, 'next tap',
                       np.where(taps['is_last'] & plausible_return, 'return home', 'unallocated'))
taps['continues'] = (~taps['is_last']) & plausible_next   # this leg chains into the next tap as the same journey

prev_continues = g['continues'].shift(1)
taps['new_journey'] = prev_continues.isna() | (prev_continues == False)  # noqa: E712 (explicit False test against a nullable-adjacent bool series)
taps['journey_no'] = taps.groupby(['CardIDbi', 'TransactionDate'])['new_journey'].cumsum()

gs = taps.drop_duplicates(['CardIDbi', 'TransactionDate'])['group_size'].value_counts(normalize=True).sort_index()
print(f"card-date groups by size (share of groups): 1 tap {gs.get(1, 0):.1%}, 2 {gs.get(2, 0):.1%}, 3 {gs.get(3, 0):.1%}, 4+ {gs[gs.index >= 4].sum():.1%} "
      f"-- a single tap in the 06:00-08:59 window (one leg, no observed transfer or return) cannot have its alighting inferred by this method at all")
unalloc_share = 1 - taps['alight_TAZ'].notna().mean()
chained_transfer_share = 1 - taps['new_journey'].mean()
tag_transfer_share = (taps['JourneyTransfer'].str.strip() == 'מעבר').mean()
print(f"taps: {len(taps):,}; alighting resolved for {taps['alight_TAZ'].notna().mean():.1%} (unallocated {unalloc_share:.1%}); "
      f"chained transfer share {chained_transfer_share:.1%} vs the file's own tag {tag_transfer_share:.1%} vs 2022's linked-journey share (~33%, step 8)")
print(taps['alight_rule'].value_counts(normalize=True).round(3).to_string())

card-date groups by size (share of groups): 1 tap 72.2%, 2 19.2%, 3 5.6%, 4+ 3.0% -- a single tap in the 06:00-08:59 window (one leg, no observed transfer or return) cannot have its alighting inferred by this method at all


taps: 6,015,350; alighting resolved for 45.8% (unallocated 54.2%); chained transfer share 27.5% vs the file's own tag 3.4% vs 2022's linked-journey share (~33%, step 8)


alight_rule
unallocated    0.542
next tap       0.275
return home    0.183


## 4. Journey-level OD by TAZ, averaged over the 42 representative Tuesdays

In [6]:
jrn = taps.groupby(['CardIDbi', 'TransactionDate', 'journey_no']).agg(
    o_TAZ=('TAZ', 'first'), d_TAZ=('alight_TAZ', 'last'), n_legs=('TAZ', 'size'), origin_tag=('JourneyTransfer', 'first')).reset_index()
jrn['allocated'] = jrn['d_TAZ'].notna()
n_journeys = len(jrn); n_alloc = int(jrn['allocated'].sum())
print(f"{n_journeys:,} journeys chained ({n_journeys / len(DATES):,.0f} per representative Tuesday); "
      f"{n_alloc:,} with an allocated destination ({n_alloc / n_journeys:.1%}); mean legs per journey {jrn['n_legs'].mean():.2f}")

od = jrn[jrn['allocated']].groupby(['o_TAZ', 'd_TAZ']).size().div(len(DATES)).rename('journeys_2025_own_alightings').reset_index()
od.to_csv(f'{OUT}/bus_od_taz_2025_own_alightings.csv', index=False, float_format='%.3f')

summary = pd.DataFrame([{'taps': len(taps) / len(DATES), 'legs (= taps, net boardings)': len(taps) / len(DATES),
                         'journeys (chained)': n_journeys / len(DATES), 'journeys allocated': n_alloc / len(DATES),
                         'unallocated share': unalloc_share, 'chained transfer share': chained_transfer_share,
                         'file tag transfer share': tag_transfer_share, 'representative Tuesdays': len(DATES)}])
summary.to_csv(f'{OUT}/journeys_2025_summary.csv', index=False, float_format='%.4f')
summary.round(3)

4,363,323 journeys chained (103,889 per representative Tuesday); 1,146,776 with an allocated destination (26.3%); mean legs per journey 1.38


,taps,"legs (= taps, net boardings)",journeys (chained),journeys allocated,unallocated share,chained transfer share,file tag transfer share,representative Tuesdays
0,143222.619,143222.619,103888.643,27304.19,0.542,0.275,0.034,42


## 5. Against the 2022 pattern and step 34's borrowed-pattern OD

In [7]:
ref22 = pd.read_csv('Output/bus/bus_od_taz_avg.csv', index_col=0); ref22.columns = ref22.columns.astype(int); ref22.index = ref22.index.astype(int)
borrowed25 = pd.read_csv(f'{OUT}/bus_od_taz_2025.csv') if os.path.exists(f'{OUT}/bus_od_taz_2025.csv') else None

def to_matrix(long_df, val):
    m = long_df.pivot_table(index='o_TAZ' if 'o_TAZ' in long_df else 'o', columns='d_TAZ' if 'd_TAZ' in long_df else 'd', values=val, aggfunc='sum')
    return m

own_m = to_matrix(od, 'journeys_2025_own_alightings')
common_o = sorted(set(own_m.index) & set(ref22.index)); common_d = sorted(set(own_m.columns) & set(ref22.columns))
a = own_m.reindex(index=common_o, columns=common_d).fillna(0).values.ravel()
b = ref22.reindex(index=common_o, columns=common_d).fillna(0).values.ravel()
mask = (a > 0) | (b > 0)
cos = float(np.dot(a[mask], b[mask]) / (np.linalg.norm(a[mask]) * np.linalg.norm(b[mask]) + 1e-9))
print(f"2025 own-alighting journey OD vs the 2022 RavKav journey OD (bus_od_taz_avg.csv), on the {len(common_o)}x{len(common_d)} common TAZs: "
      f"cosine similarity {cos:.3f} (step 35 found the 2022 RavKav OD vs the survey at cosine 0.89 on this same kind of test)")

row_sums_check = own_m.sum(axis=1)
step34_origins = pd.read_csv(f'{OUT}/boardings_by_taz_2025.csv') if os.path.exists(f'{OUT}/boardings_by_taz_2025.csv') else None
print(f"row sums of the allocated OD: {row_sums_check.sum():,.0f} journeys/day total")

fig, ax = plt.subplots(figsize=(9, 5))
d_km = haversine_km(taps['lat'], taps['lon'], taps.groupby(['CardIDbi', 'TransactionDate'])['lat'].transform('first'), taps.groupby(['CardIDbi', 'TransactionDate'])['lon'].transform('first'))
ax.hist(taps.loc[taps['alight_rule'] == 'next tap', 'dist_next_km'], bins=60, range=(0, 20), color=BLUE, alpha=0.85)
ax.set_xlabel('inferred alighting distance from the boarding stop (km, next-tap rule only)'); ax.set_ylabel('taps')
ax.set_title('RavKav 2025 own-alighting inference: leg length distribution (task C9)', loc='left', fontsize=10)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.grid(color=GRID, lw=0.6, axis='y')
fig.savefig('Output/figures/ravkav_2025_own_alighting_distance.png', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()

2025 own-alighting journey OD vs the 2022 RavKav journey OD (bus_od_taz_avg.csv), on the 707x698 common TAZs: cosine similarity 0.138 (step 35 found the 2022 RavKav OD vs the survey at cosine 0.89 on this same kind of test)
row sums of the allocated OD: 27,304 journeys/day total


## Findings

- **Most cards tap once in the 06:00–08:59 window.** 72.2% of card-date groups have exactly one
  located tap; 19.2% have two, 8.6% three or more. A single tap is one leg with no observed
  transfer or return — there is nothing in the data to infer an alighting from, whatever the
  method. This is a property of the data (an AM-only extract with no linked-journey field), not
  of the chaining method below.
- **Alighting resolved for 45.8% of taps** (unallocated 54.2%) — well below the ≥ 85 % this
  item's own check asked for. The gap is almost entirely the single-tap cards above: of the
  taps with a second tap that day to chain against, the large majority *do* resolve (27.5 % by
  the next-tap rule, 18.3 % by the return-home rule; only a small residual fails the 90-minute /
  20 km plausibility bounds). The check as written assumed most taps chain against another tap
  the same morning; most do not, in an AM-only file.
- **The chained transfer share (27.5%) sits far closer to 2022's own linked-journey rate
  (a third of legs, step 8) than the file's own `JourneyTransfer` tag does (3.4%).** This does
  not depend on alighting resolution at all — it only asks whether a tap continues a chain — and
  is the more robust finding of this notebook: it supports the standing hypothesis
  (`METHODOLOGY.md` §6af "Re-anchoring", caveat 16) that the tag marks a fare-rule transfer, not
  a physical one, and that a real transfer rate for this file is closer to 2022's than the tag
  suggests.
- **The resulting journey OD does not resemble the 2022 pattern (cosine 0.138)** — but the two
  are not comparable populations: only 26.3% of chained journeys have an allocated destination
  here, and they are systematically the journeys with an observed transfer or a same-morning
  return, not a representative sample of all AM travel the way the 2022 linked-journey file is.
  This comparison is not evidence against the 2025 data or the chaining method; it is evidence
  that this specific construction (single-morning-window tap chaining) cannot stand in for a
  general alighting inference the way step 8's provider-supplied journey IDs could in 2022.
- **What this settles for the re-anchoring question (§6af):** reason (ii) of that section — "a
  2025 alighting inference... has not been done" — is now attempted, and the attempt shows *why*
  it is hard from this file alone: an AM-only extract with mostly single-tap cards cannot supply
  a general alighting inference by tap-chaining. A full alighting inference (the kind 2022's
  RavKav data or a proper AVL/schedule-matching method provides) is still what re-anchoring on
  2025 would need; this notebook is that finding, not a working substitute for it.

**Outputs.** `Output/ravkav_2025/bus_od_taz_2025_own_alightings.csv` (the allocated journeys
only, average per representative Tuesday), `journeys_2025_summary.csv`, figure
`ravkav_2025_own_alighting_distance.png`.

**Limits.** The "same line direction" refinement of the plan's alighting rule was not built (no
line-shape matching here); the 90-minute / 20 km bounds do double duty for both the alighting
inference and the chain decision, since no separate fare-transfer-window figure exists elsewhere
in this repository; `PassengersNumber <= 0` (refund) rows are dropped before chaining, matching
the convention used elsewhere for these extracts.